# 04 - Multi-Asset Robustness

Run the same walk-forward setup on five broad ETFs spanning different asset classes:

| Ticker | Asset class | Why include it |
|---|---|---|
| SPY | US large-cap equity | The baseline |
| QQQ | US tech | Higher vol, stronger trends |
| IWM | US small-cap | Choppier, weaker drift |
| GLD | Gold | Negative correlation to equity, classic trend-follower target |
| TLT | Long Treasuries | Bond regime, very different vol structure |

Picking five ETFs (not five surviving stocks) deliberately avoids survivorship bias.

In [ ]:
import pandas as pd

from ma_backtester.config import DEFAULT_SWEEP, DEFAULT_TICKERS, CostConfig, WalkForwardConfig
from ma_backtester.costs import FixedBpsCost
from ma_backtester.data import load_close
from ma_backtester.metrics import cagr, max_drawdown, sharpe_ratio
from ma_backtester.plotting import install_theme
from ma_backtester.walk_forward import run_walk_forward

install_theme()

In [ ]:
cost = FixedBpsCost(CostConfig(per_side_bps=5.0))
wf_config = WalkForwardConfig(train_years=5, test_years=1, step_years=1)

rows = []
fold_records = []
for tk in DEFAULT_TICKERS:
    close = load_close(tk, start="2005-01-01", end="2024-12-31")
    wf = run_walk_forward(
        close=close,
        ticker=tk,
        sweep=DEFAULT_SWEEP,
        wf_config=wf_config,
        cost_model=cost,
    )
    if wf.concatenated_returns is None:
        continue
    rows.append(
        {
            "ticker": tk,
            "n_folds": len(wf.folds),
            "OOS_sharpe": sharpe_ratio(wf.concatenated_returns),
            "OOS_CAGR": cagr(wf.concatenated_equity),
            "OOS_max_DD": max_drawdown(wf.concatenated_equity),
            "pct_profitable_folds": sum(f.out_of_sample_return > 0 for f in wf.folds)
            / len(wf.folds),
        }
    )
    fold_records.extend(
        [
            {
                "ticker": tk,
                "fold": f.fold_index,
                "fast": f.selected_fast,
                "slow": f.selected_slow,
                "IS_sharpe": f.in_sample_sharpe,
                "OOS_sharpe": f.out_of_sample_sharpe,
            }
            for f in wf.folds
        ]
    )

summary = pd.DataFrame(rows).set_index("ticker")
summary

## Reading the table

Trend-following theory predicts the strategy works better on assets with persistent regimes and weaker unconditional drift. Empirically: GLD and TLT tend to be the friendliest; SPY and QQQ the hardest (because B&H is already very hard to beat in a long bull regime). IWM is choppy and the result is usually middling.

## OOS Sharpe per ticker

In [ ]:
import plotly.graph_objects as go

fig = go.Figure(go.Bar(x=summary.index, y=summary["OOS_sharpe"]))
fig.update_layout(
    title="Out-of-sample Sharpe by ticker",
    xaxis_title="ticker",
    yaxis_title="OOS Sharpe",
)
fig

In [ ]:
fold_df = pd.DataFrame(fold_records)
fold_df.groupby("ticker")[["IS_sharpe", "OOS_sharpe"]].describe()